In [ ]:
import importlib
import os
import pickle
from pathlib import Path

import gnss_tools.signals.gps_l1ca as gps_l1ca
import gnss_tools.signals.gps_l2c as gps_l2c
import matplotlib.pyplot as plt
import numpy as np
import scipy.signal
import utils.bpsk_correlation as bpsk_correlation

import utils
from utils import bpsk_acquisition, collect_metadata_utils, sample_streaming, tracking_channel
from utils.signal_interfaces import TRACKING_POLICIES, GpsL1CA, GpsL2C, build_signals

plt.rcParams.update({'font.size': 16})

In [ ]:
# Signal definitions for each potential signal we might want to test.
#
# These carry the code topology (which codes exist and how they occupy the chip
# clock) as data, so the same correlator handles L1 C/A's single code and L2C's
# interleaved CM/CL pair.  See utils/code_components.py.
GPS_L1CA_signals = build_signals(GpsL1CA)
GPS_L2C_signals = build_signals(GpsL2C)

In [ ]:
# Note: you will need to change the filepath to point to where your data is stored
#  I have mine stored in a "local-data" folder within the project directory
collects_dir = utils.environment_variables.get_collects_dir()
available_experiment_names = sorted(map(lambda fp: fp.name, collects_dir.iterdir()))
print("Available experiments:", ", ".join([str(name) for name in available_experiment_names]))
experiment_name = available_experiment_names[3]
experiment_dir = collects_dir / experiment_name
# Note: I keep a metadata.yml file in each experiment directory to keep track of sample metadata
#  and what are the available collect filenames.  You can either create your own metadata.yml,
#  or modify the code below to directly chose your data filepath and set sample parameters.
metadata_filepath = experiment_dir / "metadata.yml"
metadata = collect_metadata_utils.load_experiment_metadata_from_file(metadata_filepath, print_summary=True)

# L1
# collect_id = metadata.collect_ids[2]
# band_id = metadata.band_ids[0]
# signals = GPS_L1CA_signals
# L2
collect_id = metadata.collect_ids[1]
band_id = metadata.band_ids[1]
signals = GPS_L2C_signals

collect_config = metadata.collects[collect_id]
channel_id = collect_config.channel_config_id
band_config = metadata.band_configurations[band_id]
channel_config = metadata.channel_configurations[channel_id]

inter_freq_hz = band_config.inter_freq
samp_rate = channel_config.samp_rate
sample_params = channel_config.sample_params
collect_filepath = experiment_dir / collect_config.filename

In [10]:
# Load acquisition results from file
acq_results_directory = local_data_dir / "acquisition-results"
acq_results_directory.mkdir(parents=True, exist_ok=True)
acq_results_version_id = "v1"
acq_results_filepath = acq_results_directory / f"{collect_id}.{acq_results_version_id}.pkl"
with open(acq_results_filepath, "rb") as f:
    acq_results: dict[str, bpsk_acquisition.AcquisitionResult] = pickle.load(f)

acquired_signal_ids = sorted(list(filter(
    lambda sig_id: acq_results[sig_id].signal_detected, acq_results.keys()
)))
print(f"Acquired signals in collect {collect_id}: ")
print(", ".join(acquired_signal_ids))

NameError: name 'local_data_dir' is not defined

In [ ]:
buffer_duration_ms = 500
buffer_size_samples = int(samp_rate * buffer_duration_ms / 1e3)

sig_id = acquired_signal_ids[0]
# sig_id = "G01"
sig_def = signals[sig_id]
print(f"Example acquired signal ID: {sig_id}")
print(f"  Components: {sig_def.component_names}")
print(f"Acquisition results for signal {sig_id}:")
acq_result = acq_results[sig_id]
print(f"  Signal detected: {acq_result.signal_detected}")
print(f"  Acquisition code phase (ms): {acq_result.acq_code_phase_seconds * 1e3:.3f}")
print(f"  Acquisition Doppler (Hz): {acq_result.acq_doppler_hz:.2f}")

# Delay bins are now an explicit tuple of chip offsets rather than a count plus a
# step, so layouts other than symmetric early/prompt/late are expressible.
correlator = tracking_channel.AlignedCorrelator(
    tracking_channel.DelayDopplerCorrelatorConfig(
        bin_offsets_chips=np.array([-0.5, 0.0, 0.5]),
    ),
    num_components=sig_def.code_set.num_components,
)
acq_signal_state = tracking_channel.TrackingSignalState(
    uptime_epoch_ms=acq_result.uptime_epoch_ms,
    code_phase_ms=acq_result.acq_code_phase_seconds * 1e3,
    code_rate_ms_per_sec=(1.0 + acq_result.acq_doppler_hz / sig_def.carrier_freq_hz) * 1e3,
    carrier_phase_cycles=0.0,
    carrier_rate_cyc_per_sec=acq_result.acq_doppler_hz,
)
corr_interval = tracking_channel.CorrelationInterval(
    int(acq_signal_state.code_phase_ms + 1),
    1
)

In [ ]:
# Performed aligned correlations based on acquisition results
num_correlations = 200
correlations = np.zeros(num_correlations, dtype=complex)
corr_uptimes_ms = np.zeros(num_correlations)

# Correlate against the loop-driving component (CA for L1 C/A, CM for L2C).
driving_component = TRACKING_POLICIES[type(sig_def).signal_type_id].discriminator_policy.carrier_component

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
 ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator()
    buffer_samples = next(sample_buffer_generator)
    buffer_uptime_epoch_ms = 0.0
    sample_buffer = sample_streaming.SampleBuffer(buffer_samples, buffer_uptime_epoch_ms, samp_rate)
    print(f"Sample buffer uptime epoch: {sample_buffer.start_uptime_ms:.2f} ms")
    
    for i in range(num_correlations):
        approx_corr_start_uptime_ms, approx_corr_stop_uptime_ms = corr_interval.compute_start_and_stop_uptime_ms(acq_signal_state)
        corr_start_buffer_sample_index = int(approx_corr_start_uptime_ms * 1e-3 * samp_rate)
        corr_stop_buffer_sample_index = int(approx_corr_stop_uptime_ms * 1e-3 * samp_rate)
        actual_corr_start_uptime_ms = corr_start_buffer_sample_index / samp_rate * 1e3
        actual_corr_stop_uptime_ms = corr_stop_buffer_sample_index / samp_rate * 1e3
        corr_start_code_phase_ms, corr_start_carr_phase_cyc = acq_signal_state.propagate_phase(actual_corr_start_uptime_ms)

        corr_doppler_hz = acq_signal_state.carrier_rate_cyc_per_sec
        corr_code_phase_chips = corr_start_code_phase_ms / 1e3 * sig_def.tracking_code_rate_chips_per_sec
        corr_code_rate_chips_per_sec = acq_signal_state.code_rate_ms_per_sec / 1e3 * sig_def.tracking_code_rate_chips_per_sec

        # One prompt bin, all components; output is (num_bins, num_components).
        corr_result = np.zeros((1, sig_def.code_set.num_components), dtype=np.complex64)
        bpsk_correlation.correlate__multicomponent(
                sample_buffer.samples[corr_start_buffer_sample_index:corr_stop_buffer_sample_index],
                sample_buffer.samp_rate,
                corr_start_carr_phase_cyc,
                corr_doppler_hz,
                sig_def.code_set,
                corr_code_rate_chips_per_sec,
                corr_code_phase_chips,
                np.array([0.0]),
                corr_result,
            )
        correlations[i] = corr_result[0, driving_component]
        corr_uptimes_ms[i] = actual_corr_start_uptime_ms

        corr_interval.increment()

unwrapped_corr_phase_cyc = np.unwrap(np.angle(correlations)) / (2 * np.pi)
approx_excess_doppler_hz = (unwrapped_corr_phase_cyc[-1] - unwrapped_corr_phase_cyc[0]) / (corr_uptimes_ms[-1] - corr_uptimes_ms[0]) * 1e3
print(f"Approx. excess Doppler from correlation phase: {approx_excess_doppler_hz:.2f} Hz")

excess_doppler_phase_trend_cyc = np.polyval(np.polyfit(corr_uptimes_ms * 1e-3, unwrapped_corr_phase_cyc, 1), corr_uptimes_ms * 1e-3)

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)
ax.plot(corr_uptimes_ms, np.abs(correlations), marker="o", color="#111111")
ax.scatter(corr_uptimes_ms, correlations.real, color="r")
ax.scatter(corr_uptimes_ms, correlations.imag, color="b")
ax2 = ax.twinx()    
ax2.plot(corr_uptimes_ms, unwrapped_corr_phase_cyc - excess_doppler_phase_trend_cyc, marker="o", color="#aaaaaa")
ax.grid()
# ax.set_ylim(-20e3, 20e3)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Correlation Magnitude")
ax2.set_ylabel("Correlation Phase [cycles]")
plt.show()

In [ ]:
num_correlations = 1023 * 40
correlations = np.zeros(num_correlations, dtype=complex)
corr_phase_offset_ms = np.zeros(num_correlations)

with sample_streaming.FileSampleStream(
    collect_filepath,
    sample_params,
    buffer_size_samples,
 ) as sample_stream:

    sample_buffer_generator = sample_stream.sample_buffer_generator()
    buffer_samples = next(sample_buffer_generator)
    buffer_uptime_epoch_ms = 0.0
    sample_buffer = sample_streaming.SampleBuffer(buffer_samples, buffer_uptime_epoch_ms, samp_rate)

    for i in range(num_correlations):

        corr_phase_offset_ms[i] = i / sig_def.tracking_code_rate_chips_per_sec * 1e3
        corr_code_phase_ms = acq_result.acq_code_phase_seconds * 1e3 + corr_phase_offset_ms[i]

        corr_carr_phase_cycles = acq_signal_state.carrier_phase_cycles
        corr_doppler_hz = acq_signal_state.carrier_rate_cyc_per_sec
        corr_code_phase_chips = corr_code_phase_ms / 1e3 * sig_def.tracking_code_rate_chips_per_sec
        corr_code_rate_chips_per_sec = acq_signal_state.code_rate_ms_per_sec / 1e3 * sig_def.tracking_code_rate_chips_per_sec

        corr_result = np.zeros((1, sig_def.code_set.num_components), dtype=np.complex64)
        bpsk_correlation.correlate__multicomponent(
                sample_buffer.samples[0:50000],
                sample_buffer.samp_rate,
                corr_carr_phase_cycles,
                corr_doppler_hz,
                sig_def.code_set,
                corr_code_rate_chips_per_sec,
                corr_code_phase_chips,
                np.array([0.0]),
                corr_result,
            )
        correlations[i] = corr_result[0, driving_component]

In [ ]:
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(1, 1, 1)
ax.plot(corr_phase_offset_ms, np.abs(correlations), marker="o", color="#111111", alpha=0.1)
ax.scatter(corr_phase_offset_ms, correlations.real, color="r")
ax.scatter(corr_phase_offset_ms, correlations.imag, color="b")
ax2 = ax.twinx()
# ax2.plot(corr_phase_offset_ms, np.unwrap(np.angle(correlations)), marker="o", color="#aaaaaa")
ax.grid()
# ax.set_ylim(-20e3, 20e3)
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Correlation Magnitude")
ax2.set_ylabel("Correlation Phase [cycles]")
# ax.set_xlim(.9, 1.1)
# ax.set_xlim(3.55, 3.65)
plt.show()